# Question 2 – Cyberbullying Comment Binary Classification

**Course:** Data Management & Visualisation — CBS MSc BADS 2025–2027  

---

### Task overview
Build two supervised NLP classifiers to detect cyberbullying in social-media comments:
1. **Multinomial Naive Bayes** (scikit-learn)
2. **Logistic Regression** (scikit-learn)

### Pipeline
```
Load data → Explore → Preprocess → TF-IDF vectorise → Train/Test split
         → Naive Bayes → Evaluate
         → Logistic Regression → Evaluate
         → Compare & Report
```

## Imports

In [ ]:
import re                         # regular expressions for text cleaning
import warnings

import numpy  as np
import pandas as pd

# scikit-learn: splitting, vectorisation, models, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import (accuracy_score,
                                              classification_report,
                                              confusion_matrix,
                                              f1_score,
                                              precision_score,
                                              recall_score)
from sklearn.model_selection         import train_test_split
from sklearn.naive_bayes             import MultinomialNB

warnings.filterwarnings("ignore")   # suppress convergence / version warnings

---
## Part 1 — Data Preparation

### 1-A · Load the dataset

The CSV contains two columns:
| Column | Type | Description |
|--------|------|-------------|
| `Text` | str | Raw social-media comment |
| `CB_Label` | int | **1** = cyberbullying · **0** = not cyberbullying |

In [ ]:
# Read the CSV — place the file in the same directory as this notebook
df = pd.read_csv("CyberBullying_Comments_Dataset.csv")

print("Shape:", df.shape)
df.head()

### 1-B · Basic data exploration

In [ ]:
# Data types and missing values
print("Data types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Class distribution — are the classes balanced?
label_counts = df["CB_Label"].value_counts()
label_pct    = df["CB_Label"].value_counts(normalize=True) * 100

print("Label counts:")
print(label_counts.rename({0: "Not Cyberbullying (0)", 1: "Cyberbullying (1)"}))

print("\nLabel percentages:")
print(label_pct.rename({0: "Not Cyberbullying (0)", 1: "Cyberbullying (1)"}).round(2))

# A perfectly balanced 50/50 split means accuracy is a reliable metric
# and no class-weight adjustments or oversampling (e.g. SMOTE) are needed.

### 1-C · Text preprocessing

Raw social-media text is noisy — we apply the following cleaning steps **in order**:

| Step | Operation | Reason |
|------|-----------|--------|
| 1 | Lowercase | Normalise `"HATE"` and `"hate"` to the same token |
| 2 | Remove URLs | Hyperlinks carry no semantic value |
| 3 | Keep letters only | Strip punctuation, digits, emojis |
| 4 | Collapse whitespace | Tidy up residual spaces |
| 5 | Remove stop words | Common words (`"the"`, `"is"`) add no discriminative signal |
| 6 | Drop short tokens | Tokens of length ≤ 2 are typically noise |

> **Note:** NLTK is not required — we use a hand-curated stop-word list covering the standard English corpus plus Twitter-specific tokens (`rt`, `via`, `amp`).

In [ ]:
# ── Stop-word list (English + Twitter-specific tokens) ──────────────────────
STOP_WORDS = frozenset([
    "i","me","my","myself","we","our","ours","ourselves","you","your","yours",
    "yourself","yourselves","he","him","his","himself","she","her","hers",
    "herself","it","its","itself","they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those","am","is","are",
    "was","were","be","been","being","have","has","had","having","do","does",
    "did","doing","a","an","the","and","but","if","or","because","as","until",
    "while","of","at","by","for","with","about","against","between","into",
    "through","during","before","after","above","below","to","from","up","down",
    "in","out","on","off","over","under","again","further","then","once","here",
    "there","when","where","why","how","all","both","each","few","more","most",
    "other","some","such","no","nor","not","only","own","same","so","than",
    "too","very","s","t","can","will","just","don","should","now","d","ll",
    "m","o","re","ve","y","ain","aren","couldn","didn","doesn","hadn","hasn",
    "haven","isn","ma","mightn","mustn","needn","shan","shouldn","wasn",
    "weren","won","wouldn",
    # Twitter / social-media specific
    "rt", "via", "amp",
])


def preprocess(text: str) -> str:
    """
    Clean and tokenise a single comment string.

    Parameters
    ----------
    text : str
        Raw social-media comment.

    Returns
    -------
    str
        Space-joined list of clean tokens, ready for TF-IDF vectorisation.
    """
    # Step 1 — lowercase the entire string
    text = str(text).lower()

    # Step 2 — remove URLs (http://…, https://…, www.…)
    text = re.sub(r"http\S+|www\S+", "", text)

    # Step 3 — keep only ASCII letters and spaces
    text = re.sub(r"[^a-z\s]", "", text)

    # Step 4 — collapse multiple consecutive spaces to one
    text = re.sub(r"\s+", " ", text).strip()

    # Steps 5 & 6 — drop stop words and tokens of length ≤ 2
    tokens = [
        token for token in text.split()
        if token not in STOP_WORDS and len(token) > 2
    ]

    # Rejoin into a single string (required by TfidfVectorizer)
    return " ".join(tokens)

In [ ]:
# Apply preprocessing to every comment
df["clean_text"] = df["Text"].apply(preprocess)

# Demonstrate the effect on a sample row
idx = 10
print("Original :", df["Text"].iloc[idx])
print("Cleaned  :", df["clean_text"].iloc[idx])

In [ ]:
# After cleaning, some comments may become empty strings
# (e.g. a tweet that was purely a URL) — drop them.
empty_mask = df["clean_text"].str.strip() == ""
print(f"Rows emptied by preprocessing: {empty_mask.sum()} — dropping them.")

df = df[~empty_mask].reset_index(drop=True)
print(f"Usable rows after filtering  : {len(df)}")

### 1-D · Train / Test split (80 % / 20 %)

> **Important:** we split *before* fitting the vectoriser to avoid **data leakage** — the TF-IDF vocabulary and IDF weights must be learned only from training data.

In [ ]:
X = df["clean_text"]   # features: cleaned text strings
y = df["CB_Label"]     # target: 0 or 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,   # 20 % held out for testing
    random_state = 42,     # fixed seed → fully reproducible
    stratify     = y,      # preserve the 50/50 class ratio in both splits
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"\nTrain label dist :\n{y_train.value_counts()}")
print(f"\nTest  label dist :\n{y_test.value_counts()}")

### 1-E · TF-IDF Vectorisation

**TF-IDF** (Term Frequency–Inverse Document Frequency) weights each n-gram by how informative it is:
- **High weight** → appears often in *this* document but rarely across all documents (discriminative)
- **Low weight** → appears in almost every document (not useful for classification)

| Hyperparameter | Value | Reason |
|---|---|---|
| `max_features` | 10 000 | Cap vocabulary size; removes long-tail noise |
| `ngram_range` | (1, 2) | Include bigrams — phrases like *"shut up"* carry meaning that unigrams alone miss |
| `min_df` | 2 | Ignore n-grams appearing in fewer than 2 documents (likely typos) |
| `sublinear_tf` | True | Replace raw TF with `1 + log(TF)` — reduces the dominance of very frequent terms |

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features = 10_000,
    ngram_range  = (1, 2),   # unigrams + bigrams
    min_df       = 2,        # minimum document frequency
    sublinear_tf = True,     # log-scaled TF
)

# fit_transform on train — learns vocabulary & IDF weights from training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# transform on test — applies the *same* vocabulary/IDF (no leakage)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF feature matrix — train: {X_train_tfidf.shape}")
print(f"TF-IDF feature matrix — test : {X_test_tfidf.shape}")

---
## Part 2 — Naive Bayes Classification

We choose **Multinomial Naive Bayes (MNB)** because our features are TF-IDF scores — non-negative, count-like values that fit the multinomial distribution assumption.

| Variant | Best for | Our choice? |
|---------|----------|-------------|
| `MultinomialNB` | TF-IDF / word-count features | ✅ |
| `BernoulliNB` | Binary word-presence features | ✗ |
| `GaussianNB` | Continuous numerical features | ✗ |

**`alpha=0.1`** applies Laplace smoothing — a small pseudo-count added to every feature so that n-grams unseen during training don't produce a zero probability at inference time. The default `alpha=1.0` over-smooths on larger datasets; `0.1` is a lighter touch.

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
nb_model = MultinomialNB(alpha=0.1)   # alpha: Laplace smoothing parameter
nb_model.fit(X_train_tfidf, y_train)

# ── Predict ──────────────────────────────────────────────────────────────────
y_pred_nb = nb_model.predict(X_test_tfidf)

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
nb_accuracy  = accuracy_score(y_test, y_pred_nb)
nb_precision = precision_score(y_test, y_pred_nb)  # positive class = 1 (CB)
nb_recall    = recall_score(y_test, y_pred_nb)
nb_f1        = f1_score(y_test, y_pred_nb)
nb_cm        = confusion_matrix(y_test, y_pred_nb)

print("Naive Bayes — Test-Set Performance")
print(f"  Accuracy  : {nb_accuracy:.4f}")
print(f"  Precision : {nb_precision:.4f}   (of all predicted CB, how many are truly CB?)")
print(f"  Recall    : {nb_recall:.4f}   (of all true CB, how many did we catch?)")
print(f"  F1-Score  : {nb_f1:.4f}   (harmonic mean of precision and recall)")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_nb,
                             target_names=["Not Cyberbullying", "Cyberbullying"]))

print("Confusion matrix (rows = actual, cols = predicted):")
cm_df = pd.DataFrame(
    nb_cm,
    index   = ["Actual: Not CB", "Actual: CB"],
    columns = ["Pred: Not CB",   "Pred: CB"]
)
print(cm_df)
print(f"\n  TN={nb_cm[0,0]}  FP={nb_cm[0,1]}  FN={nb_cm[1,0]}  TP={nb_cm[1,1]}")

---
## Part 3 — Logistic Regression Classification

**Logistic Regression (LR)** is a *discriminative* linear classifier that models `P(y=1 | X)` directly via the logistic sigmoid, rather than modelling the joint distribution like Naive Bayes.

Key differences vs. Naive Bayes:
- **No independence assumption** — weights are jointly optimised across all features
- **L2 regularisation** applied by default, penalising large coefficients to reduce overfitting
- **Interpretable** — the sign and magnitude of coefficients reveal which words drive each class

| Hyperparameter | Value | Reason |
|---|---|---|
| `C` | 1.0 | Inverse regularisation strength — default, moderate penalty |
| `solver` | `lbfgs` | Efficient for medium-sized dense problems; supports L2 natively |
| `max_iter` | 1000 | Default 100 often insufficient for high-dimensional TF-IDF matrices |
| `random_state` | 42 | Reproducibility |

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
lr_model = LogisticRegression(
    C            = 1.0,      # inverse regularisation strength
    solver       = "lbfgs",  # Limited-memory BFGS optimiser
    max_iter     = 1000,     # iterations until convergence
    random_state = 42,
)
lr_model.fit(X_train_tfidf, y_train)

# ── Predict ──────────────────────────────────────────────────────────────────
y_pred_lr = lr_model.predict(X_test_tfidf)

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
lr_accuracy  = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall    = recall_score(y_test, y_pred_lr)
lr_f1        = f1_score(y_test, y_pred_lr)
lr_cm        = confusion_matrix(y_test, y_pred_lr)

print("Logistic Regression — Test-Set Performance")
print(f"  Accuracy  : {lr_accuracy:.4f}")
print(f"  Precision : {lr_precision:.4f}   (of all predicted CB, how many are truly CB?)")
print(f"  Recall    : {lr_recall:.4f}   (of all true CB, how many did we catch?)")
print(f"  F1-Score  : {lr_f1:.4f}   (harmonic mean of precision and recall)")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_lr,
                             target_names=["Not Cyberbullying", "Cyberbullying"]))

print("Confusion matrix (rows = actual, cols = predicted):")
cm_df = pd.DataFrame(
    lr_cm,
    index   = ["Actual: Not CB", "Actual: CB"],
    columns = ["Pred: Not CB",   "Pred: CB"]
)
print(cm_df)
print(f"\n  TN={lr_cm[0,0]}  FP={lr_cm[0,1]}  FN={lr_cm[1,0]}  TP={lr_cm[1,1]}")

---
## Part 4 — Model Comparison & Short Report

In [ ]:
# ── Side-by-side comparison table ────────────────────────────────────────────
comparison = pd.DataFrame({
    "Naive Bayes":        [nb_accuracy, nb_precision, nb_recall, nb_f1],
    "Logistic Regression": [lr_accuracy, lr_precision, lr_recall, lr_f1],
}, index=["Accuracy", "Precision (CB class)", "Recall (CB class)", "F1-Score (CB class)"])

# Mark the winning model for each metric
comparison["Winner"] = comparison.apply(
    lambda row: "NB" if row["Naive Bayes"] > row["Logistic Regression"]
               else ("LR" if row["Logistic Regression"] > row["Naive Bayes"] else "Tie"),
    axis=1
)

print(comparison.round(4).to_string())

### Overall performance

Both models achieve **~71 % accuracy** on a balanced test set, suggesting the task has inherent difficulty for linear bag-of-words approaches.

- **Naive Bayes** leads on accuracy, recall, and F1-score.
- **Logistic Regression** leads on precision for the cyberbullying class.

---

### Naive Bayes — Advantages

1. **Higher recall (69.6 % vs 66.2 %)** — In a harm-reduction context, failing to flag a real bullying comment (false negative) is more costly than a false alarm. Higher recall is therefore the more important metric here.
2. **Extremely fast training** — parameters are estimated analytically in a single pass; no iterative optimisation required.
3. **Robust on smaller training sets** — class-conditional word probabilities are stable estimates even with limited data.

### Naive Bayes — Disadvantages

1. **Conditional independence assumption** — every feature (n-gram) is assumed independent of all others given the class. Natural language violates this: *"not good"* is very different from *"not"* and *"good"* considered separately.
2. **Cannot capture negative correlations** — all TF-IDF weights in MNB are non-negative, so a word that strongly predicts the *absence* of cyberbullying cannot reduce the cyberbullying score.

---

### Logistic Regression — Advantages

1. **Higher precision (72.8 % vs 71.6 %)** — fewer innocent comments are falsely flagged; preferable where false accusations carry a high social cost.
2. **No independence assumption** — coefficients are jointly optimised over the entire feature space, naturally accounting for correlations between co-occurring n-grams.
3. **Interpretable weights** — inspecting the largest positive/negative coefficients reveals which words and bigrams drive each class.

### Logistic Regression — Disadvantages

1. **Lower recall (66.2 % vs 69.6 %)** — 376 true cyberbullying comments are missed versus 338 for Naive Bayes, a meaningful gap in a safety-critical application.
2. **Slower training** — iterative L-BFGS optimisation over 10 000 features is significantly more expensive than MNB's closed-form estimation.

---

### Other important findings

- **Perfect class balance (50/50)** is a rare and favourable dataset property — accuracy is a reliable summary metric and no oversampling (e.g. SMOTE) was needed.
- **Both models plateau at ~71 %**, strongly suggesting the task exceeds what linear bag-of-words models can capture. Cyberbullying often relies on sarcasm, context, code-switching and emojis — all discarded by the preprocessing pipeline. Fine-tuned transformer models (e.g. BERT) would likely yield substantially higher performance.
- **Bigrams meaningfully enrich the feature space** (`ngram_range=(1,2)`): phrases such as *"shut up"*, *"go die"* or *"kill yourself"* carry clear bullying intent that individual unigrams alone do not capture.

---

### References

1. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
2. Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
3. Hosmer, D. W., & Lemeshow, S. (2000). *Applied Logistic Regression* (2nd ed.). Wiley-Interscience.
4. Sooraj Tomar (2023). Cyberbullying Tweets Dataset. Kaggle. https://www.kaggle.com/datasets/soorajtomar/cyberbullying-tweets